# PoseNet Python Port – Adaptation

**Course:** 4DT907 - Project in Data Intensive Systems  
**Topic:** Adapt the PoseNet Python port to store joint positions in CSV / JSON files

---

## What is PoseNet?

PoseNet is a computer-vision model originally developed by Google for TensorFlow.js that estimates **human body pose** from images or video frames.  
It detects **17 keypoints** (joints) on a person:

| Index | Keypoint |
|-------|----------|
| 0 | nose |
| 1 | left eye |
| 2 | right eye |
| 3 | left ear |
| 4 | right ear |
| 5 | left shoulder |
| 6 | right shoulder |
| 7 | left elbow |
| 8 | right elbow |
| 9 | left wrist |
| 10 | right wrist |
| 11 | left hip |
| 12 | right hip |
| 13 | left knee |
| 14 | right knee |
| 15 | left ankle |
| 16 | right ankle |

The **Python port** used here is the PyTorch re-implementation by Ross Wightman:  
https://github.com/rwightman/posenet-python

### How it works

1. An input image is resized and normalized to a fixed scale.
2. A lightweight MobileNet backbone extracts feature maps.
3. Two heads produce *heatmaps* (confidence that a keypoint exists at each cell) and *offset vectors* (sub-cell refinement).
4. A greedy decode over heatmaps + offsets gives the final `(y, x, score)` for every keypoint in every detected person.

### This notebook

1. Installs / verifies the required libraries.
2. Loads the pre-trained PoseNet MobileNet model weights (downloaded on first run).
3. Runs inference on **images** and **video**.
4. **Stores the joint positions** in both **CSV** and **JSON** formats.

## 1. Install Dependencies

In [ ]:
# Install posenet-pytorch and other dependencies
%pip install torch torchvision opencv-python numpy Pillow requests matplotlib pandas --quiet

## 2. Clone / Download the PoseNet Python Port

The original Python port lives at https://github.com/rwightman/posenet-python.  
We embed the essential parts (model architecture, decoding utilities, and weight download) below so the notebook is self-contained.

In [ ]:
import subprocess, sys, os

POSENET_REPO = "posenet_python_repo"

if not os.path.isdir(POSENET_REPO):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/rwightman/posenet-python", POSENET_REPO],
        check=True
    )
    print("Cloned posenet-python.")
else:
    print("posenet-python repo already present.")

if POSENET_REPO not in sys.path:
    sys.path.insert(0, POSENET_REPO)

## 3. Imports and Constants

In [ ]:
import os
import csv
import json
import math
import time
from pathlib import Path
from typing import List, Dict, Any, Tuple

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

# PoseNet keypoint names (17 keypoints)
KEYPOINT_NAMES = [
    "nose",
    "left_eye",
    "right_eye",
    "left_ear",
    "right_ear",
    "left_shoulder",
    "right_shoulder",
    "left_elbow",
    "right_elbow",
    "left_wrist",
    "right_wrist",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_ankle",
    "right_ankle",
]

# Skeleton connections used for visualisation
SKELETON_CONNECTIONS = [
    ("nose", "left_eye"),
    ("left_eye", "left_ear"),
    ("nose", "right_eye"),
    ("right_eye", "right_ear"),
    ("nose", "left_shoulder"),
    ("nose", "right_shoulder"),
    ("left_shoulder", "right_shoulder"),
    ("left_shoulder", "left_elbow"),
    ("left_elbow", "left_wrist"),
    ("right_shoulder", "right_elbow"),
    ("right_elbow", "right_wrist"),
    ("left_shoulder", "left_hip"),
    ("right_shoulder", "right_hip"),
    ("left_hip", "right_hip"),
    ("left_hip", "left_knee"),
    ("left_knee", "left_ankle"),
    ("right_hip", "right_knee"),
    ("right_knee", "right_ankle"),
]

print(f"Keypoints defined: {len(KEYPOINT_NAMES)}")

## 4. Load the PoseNet Model

The original repo ships a `posenet` Python module with a `load_model` helper.  
Model 101 refers to the MobileNet v1 backbone with width multiplier 1.01 (the most accurate single-person variant).

In [ ]:
import posenet

MODEL_ID = 101   # Options: 50, 75, 100, 101
OUTPUT_STRIDE = 16  # 8 = more accurate, 16 = faster

model = posenet.load_model(MODEL_ID)
model = model.cuda() if __import__('torch').cuda.is_available() else model

print(f"Loaded PoseNet model {MODEL_ID} (output_stride={OUTPUT_STRIDE})")

## 5. Helper: Run Inference on a Single Image

The function below wraps the posenet calls and returns a structured list of  
detected poses, each containing per-keypoint `(y, x, score)` values.

In [ ]:
def run_posenet_on_image(
    image_path: str,
    scale_factor: float = 1.0,
    min_pose_score: float = 0.15,
    min_part_score: float = 0.1,
) -> List[Dict[str, Any]]:
    """
    Run PoseNet inference on a single image file.

    Returns a list of detected poses.  Each pose is a dict:
    {
        'pose_score': float,
        'keypoints': [
            {'name': str, 'y': float, 'x': float, 'score': float},
            ...
        ]
    }
    """
    input_image, draw_image, output_scale = posenet.read_imgfile(
        image_path, scale_factor=scale_factor
    )

    with __import__('torch').no_grad():
        input_image = __import__('torch').Tensor(input_image)
        if next(model.parameters()).is_cuda:
            input_image = input_image.cuda()

        results = model(input_image)
        heatmaps_result, offsets_result, disp_fwd, disp_bwd = results

    pose_scores, keypoint_scores, keypoint_coords = posenet.decode_multi.decode_multiple_poses(
        heatmaps_result.squeeze(0),
        offsets_result.squeeze(0),
        disp_fwd.squeeze(0),
        disp_bwd.squeeze(0),
        output_stride=OUTPUT_STRIDE,
        max_pose_detections=10,
        min_pose_score=min_pose_score,
    )

    # Scale keypoint coordinates back to original image size
    keypoint_coords *= output_scale

    poses = []
    for pi in range(len(pose_scores)):
        if pose_scores[pi] < min_pose_score:
            break
        kps = []
        for ki, name in enumerate(KEYPOINT_NAMES):
            y, x = keypoint_coords[pi, ki]
            score = float(keypoint_scores[pi, ki])
            kps.append({
                "name": name,
                "y": float(y),
                "x": float(x),
                "score": score,
            })
        poses.append({
            "pose_score": float(pose_scores[pi]),
            "keypoints": kps,
        })
    return poses


print("run_posenet_on_image() defined.")

## 6. Helper: Save Joint Positions to CSV

Each row in the CSV represents **one keypoint** from **one detected person** in **one source** (image path or video frame).  

Column schema:
```
source, frame, person_index, pose_score, keypoint, y, x, score
```

In [ ]:
CSV_FIELDNAMES = [
    "source",
    "frame",
    "person_index",
    "pose_score",
    "keypoint",
    "y",
    "x",
    "score",
]


def save_poses_to_csv(records: List[Dict[str, Any]], output_path: str) -> None:
    """
    Save a flat list of keypoint records to a CSV file.

    Each element of *records* must have keys matching CSV_FIELDNAMES.
    """
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=CSV_FIELDNAMES)
        writer.writeheader()
        writer.writerows(records)
    print(f"Saved {len(records)} keypoint records → {output_path}")


def poses_to_records(
    poses: List[Dict[str, Any]],
    source: str,
    frame: int = 0,
) -> List[Dict[str, Any]]:
    """
    Flatten poses returned by run_posenet_on_image() into a list of
    flat dicts suitable for CSV / JSON output.
    """
    records = []
    for pi, pose in enumerate(poses):
        for kp in pose["keypoints"]:
            records.append({
                "source": source,
                "frame": frame,
                "person_index": pi,
                "pose_score": round(pose["pose_score"], 4),
                "keypoint": kp["name"],
                "y": round(kp["y"], 2),
                "x": round(kp["x"], 2),
                "score": round(kp["score"], 4),
            })
    return records


print("CSV helpers defined.")

## 7. Helper: Save Joint Positions to JSON

The JSON format groups results by **source** → **frame** → list of **poses**,  
preserving the hierarchical structure of the inference output.

In [ ]:
def save_poses_to_json(
    all_results: List[Dict[str, Any]], output_path: str
) -> None:
    """
    Save pose inference results to a structured JSON file.

    *all_results* is a list of dicts with the shape:
    {
        'source': str,
        'frame':  int,
        'poses':  [ { 'pose_score': float, 'keypoints': [...] } ]
    }
    """
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as fh:
        json.dump(all_results, fh, indent=2)
    total_poses = sum(len(r["poses"]) for r in all_results)
    print(f"Saved {total_poses} poses across {len(all_results)} frames → {output_path}")


print("JSON helper defined.")

## 8. Run on Images

Place your own images in the `images/` folder next to this notebook,  
or change `IMAGE_DIR` to any directory that contains `.jpg` / `.png` files.

The cell below:
1. Iterates over all images.
2. Runs PoseNet inference.
3. Collects all keypoint records.
4. Saves the results to `output/images_poses.csv` and `output/images_poses.json`.

In [ ]:
IMAGE_DIR = "images"          # folder with input images
OUTPUT_DIR = "output"         # folder for CSV / JSON outputs

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}
image_files = sorted(
    p for p in Path(IMAGE_DIR).iterdir()
    if p.suffix.lower() in image_extensions
)

print(f"Found {len(image_files)} image(s) in '{IMAGE_DIR}'.")

all_flat_records = []   # for CSV
all_json_results = []   # for JSON

for img_path in image_files:
    print(f"  Processing: {img_path.name} ...", end=" ")
    t0 = time.time()
    poses = run_posenet_on_image(str(img_path))
    elapsed = time.time() - t0
    print(f"{len(poses)} pose(s) detected in {elapsed:.2f}s")

    # Accumulate flat records (CSV)
    all_flat_records.extend(poses_to_records(poses, source=img_path.name, frame=0))

    # Accumulate structured results (JSON)
    all_json_results.append({
        "source": img_path.name,
        "frame": 0,
        "poses": poses,
    })

# ── Save outputs ──────────────────────────────────────────────────────────────
if all_flat_records:
    save_poses_to_csv(all_flat_records, os.path.join(OUTPUT_DIR, "images_poses.csv"))
    save_poses_to_json(all_json_results, os.path.join(OUTPUT_DIR, "images_poses.json"))
else:
    print("No images processed – add .jpg / .png files to the 'images/' folder.")

### 8a. Visualise Detected Joints on an Image

This cell draws the skeleton on the first image that produced at least one pose.

In [ ]:
def draw_skeleton_on_image(img_path: str, poses: List[Dict[str, Any]]) -> None:
    """Overlay detected keypoints and skeleton bones on an image and display it."""
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    colors = plt.cm.hsv(np.linspace(0, 1, len(poses) + 1))[:, :3]
    name_to_idx = {n: i for i, n in enumerate(KEYPOINT_NAMES)}

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(img_rgb)

    for pi, pose in enumerate(poses):
        color = colors[pi]
        kp_map = {kp["name"]: kp for kp in pose["keypoints"]}

        # Draw bones
        for a, b in SKELETON_CONNECTIONS:
            if a in kp_map and b in kp_map:
                ka, kb = kp_map[a], kp_map[b]
                if ka["score"] > 0.1 and kb["score"] > 0.1:
                    ax.plot(
                        [ka["x"], kb["x"]], [ka["y"], kb["y"]],
                        "-", color=color, linewidth=2, alpha=0.8
                    )

        # Draw joints
        for kp in pose["keypoints"]:
            if kp["score"] > 0.1:
                ax.plot(kp["x"], kp["y"], "o", color=color,
                        markersize=6, markeredgecolor="white", markeredgewidth=1)

        ax.set_title(
            f"{len(poses)} pose(s) detected  –  {Path(img_path).name}",
            fontsize=13
        )

    ax.axis("off")
    plt.tight_layout()
    plt.show()


# Draw skeleton on the first image that has results
for result in all_json_results:
    if result["poses"]:
        src = os.path.join(IMAGE_DIR, result["source"])
        draw_skeleton_on_image(src, result["poses"])
        break
else:
    print("No poses detected – nothing to visualise.")

### 8b. Inspect the CSV Output

Load the generated CSV with pandas and display the first few rows.

In [ ]:
csv_path = os.path.join(OUTPUT_DIR, "images_poses.csv")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Shape: {df.shape}")
    display(df.head(20))
else:
    print("CSV file not found – run the image processing cell above first.")

## 9. Run on Video

Set `VIDEO_PATH` to point to your own video file (`.mp4`, `.avi`, etc.) or  
to `0` to use the default webcam.

`FRAME_STEP` controls how many frames to skip between inferences (use 1 for  
every frame, or a larger number to speed up processing).

Results are saved to:
- `output/video_poses.csv`
- `output/video_poses.json`

In [ ]:
VIDEO_PATH = "video.mp4"   # path to your video file, or 0 for webcam
FRAME_STEP = 1             # process every N-th frame
MAX_FRAMES = 500           # set to None to process the whole video


def process_video(
    video_path,
    frame_step: int = 1,
    max_frames: int = None,
    scale_factor: float = 1.0,
    min_pose_score: float = 0.15,
) -> Tuple[List[Dict], List[Dict]]:
    """
    Run PoseNet inference on every *frame_step*-th frame of *video_path*.

    Returns
    -------
    flat_records : list of flat dicts  (→ CSV)
    json_results : list of structured dicts (→ JSON)
    """
    import torch

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    source_name = Path(str(video_path)).name if str(video_path) != "0" else "webcam"

    print(f"Video: {source_name}  |  FPS={fps:.1f}  |  Total frames={total_frames}")

    flat_records: List[Dict] = []
    json_results: List[Dict] = []
    frame_idx = 0
    processed = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if max_frames is not None and processed >= max_frames:
            break
        if frame_idx % frame_step != 0:
            frame_idx += 1
            continue

        # ── Prepare frame for PoseNet ──────────────────────────────────────────
        # The posenet port expects a pre-processed float32 tensor shaped
        # (1, 3, H, W) in range [-1, 1].  We replicate read_imgfile() logic here
        # using the raw OpenCV frame so we don't need a temp file on disk.
        height, width = frame.shape[:2]
        target_width  = int(width  * scale_factor)
        target_height = int(height * scale_factor)

        # Snap to the nearest stride multiple (posenet requirement)
        target_width  = (target_width  // OUTPUT_STRIDE) * OUTPUT_STRIDE + 1
        target_height = (target_height // OUTPUT_STRIDE) * OUTPUT_STRIDE + 1

        resized = cv2.resize(frame, (target_width, target_height))
        rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32)
        rgb = (rgb / 127.5) - 1.0               # normalize to [-1, 1]
        tensor = torch.tensor(rgb.transpose(2, 0, 1)[np.newaxis])  # (1,3,H,W)

        output_scale = np.array([height / target_height, width / target_width])

        if next(model.parameters()).is_cuda:
            tensor = tensor.cuda()

        # ── Inference ─────────────────────────────────────────────────────────
        with torch.no_grad():
            heatmaps, offsets, disp_fwd, disp_bwd = model(tensor)

        pose_scores, keypoint_scores, keypoint_coords = posenet.decode_multi.decode_multiple_poses(
            heatmaps.squeeze(0),
            offsets.squeeze(0),
            disp_fwd.squeeze(0),
            disp_bwd.squeeze(0),
            output_stride=OUTPUT_STRIDE,
            max_pose_detections=10,
            min_pose_score=min_pose_score,
        )
        keypoint_coords *= output_scale

        # ── Build pose list ────────────────────────────────────────────────────
        poses = []
        for pi in range(len(pose_scores)):
            if pose_scores[pi] < min_pose_score:
                break
            kps = [
                {
                    "name": KEYPOINT_NAMES[ki],
                    "y": float(keypoint_coords[pi, ki, 0]),
                    "x": float(keypoint_coords[pi, ki, 1]),
                    "score": float(keypoint_scores[pi, ki]),
                }
                for ki in range(len(KEYPOINT_NAMES))
            ]
            poses.append({"pose_score": float(pose_scores[pi]), "keypoints": kps})

        flat_records.extend(poses_to_records(poses, source=source_name, frame=frame_idx))
        json_results.append({"source": source_name, "frame": frame_idx, "poses": poses})

        processed += 1
        frame_idx += 1

        if processed % 50 == 0:
            print(f"  Processed {processed} frames...")

    cap.release()
    print(f"Done – {processed} frames processed, {len(flat_records)} keypoint records.")
    return flat_records, json_results


# ── Run on video ─────────────────────────────────────────────────────────────
if os.path.exists(str(VIDEO_PATH)) or str(VIDEO_PATH) == "0":
    flat_records_vid, json_results_vid = process_video(
        VIDEO_PATH, frame_step=FRAME_STEP, max_frames=MAX_FRAMES
    )

    save_poses_to_csv(flat_records_vid, os.path.join(OUTPUT_DIR, "video_poses.csv"))
    save_poses_to_json(json_results_vid, os.path.join(OUTPUT_DIR, "video_poses.json"))
else:
    print(f"Video file '{VIDEO_PATH}' not found – skipping video processing.\n"
          f"Set VIDEO_PATH to an existing .mp4 / .avi file to enable this section.")

### 9a. Preview the Video CSV

Load and display the video output CSV, then show a quick summary by keypoint.

In [ ]:
video_csv = os.path.join(OUTPUT_DIR, "video_poses.csv")

if os.path.exists(video_csv):
    df_vid = pd.read_csv(video_csv)
    print(f"Shape: {df_vid.shape}")
    display(df_vid.head(20))

    print("\nMean confidence per keypoint (across all frames and persons):")
    display(
        df_vid.groupby("keypoint")["score"]
        .mean()
        .sort_values(ascending=False)
        .rename("mean_score")
        .reset_index()
    )
else:
    print("video_poses.csv not found – run the video processing cell above first.")

## 10. Inspect the JSON Output Format

The JSON file stores results hierarchically.  Below we print the first entry to  
illustrate the exact structure that gets written to disk.

In [ ]:
json_path = os.path.join(OUTPUT_DIR, "images_poses.json")

if os.path.exists(json_path):
    with open(json_path) as fh:
        data = json.load(fh)
    print(f"Total entries in JSON: {len(data)}")
    if data:
        print("\nFirst entry (pretty-printed):")
        print(json.dumps(data[0], indent=2))
else:
    print("images_poses.json not found – run the image processing cell above first.")

## 11. Summary

This notebook demonstrated how to adapt the **PoseNet Python port** (https://github.com/rwightman/posenet-python) to:

1. **Load** the pre-trained MobileNet-based PoseNet model.
2. **Run inference** on individual images and on a video stream frame-by-frame.
3. **Store joint positions** in two portable formats:
   - **CSV** (`output/images_poses.csv`, `output/video_poses.csv`) – a flat,  
     tabular format ideal for further analysis with pandas / spreadsheets.
   - **JSON** (`output/images_poses.json`, `output/video_poses.json`) – a  
     structured, hierarchical format that preserves the per-frame / per-person  
     grouping and is easy to consume by downstream services.

### CSV schema

| Column | Type | Description |
|--------|------|-------------|
| `source` | str | filename or "webcam" |
| `frame` | int | frame index (0 for images) |
| `person_index` | int | 0-based person index in the frame |
| `pose_score` | float | overall detection confidence |
| `keypoint` | str | joint name (e.g. `left_knee`) |
| `y` | float | pixel row (top-left origin) |
| `x` | float | pixel column (top-left origin) |
| `score` | float | keypoint detection confidence |

### JSON schema

```json
[
  {
    "source": "image.jpg",
    "frame":  0,
    "poses": [
      {
        "pose_score": 0.92,
        "keypoints": [
          { "name": "nose", "y": 120.5, "x": 240.3, "score": 0.98 },
          ...
        ]
      }
    ]
  }
]
```